In [2]:
import pandas as pd
import numpy as np
import pyarrow.parquet as pq
from pathlib import Path

BASE_DIR = Path('..')
RAW_DIR = BASE_DIR / 'data' / 'raw'
PROC_DIR = BASE_DIR / 'data' / 'processed'

# load processed files from notebook 01
ic50_matrix = pd.read_csv(PROC_DIR / 'gdsc2_ic50_matrix.csv', index_col=0)
gdsc2_matched = pd.read_csv(PROC_DIR / 'gdsc2_matched.csv')
compound_lookup = pd.read_csv(PROC_DIR / 'compound_lookup.csv')
profiles = pd.read_parquet(PROC_DIR / 'jump_profiles_matched.parquet')

print(f"IC50 matrix: {ic50_matrix.shape}")
print(f"GDSC2 matched: {gdsc2_matched.shape}")
print(f"Compound lookup: {compound_lookup.shape}")
print(f"Profiles: {profiles.shape}")

IC50 matrix: (175, 969)
GDSC2 matched: (153882, 19)
Compound lookup: (175, 4)
Profiles: (18970, 3184)


In [3]:
#  aggregate replicates into one consensus profile per drug

meta_cols = [c for c in profiles.columns if c.startswith('Metadata_')]
feature_cols = [c for c in profiles.columns if not c.startswith('Metadata_')]

print(f"Metadata columns: {len(meta_cols)}")
print(f"Feature columns:  {len(feature_cols)}")

# Replicates per compound
replicate_counts = profiles.groupby('Metadata_JCP2022').size()
print(f"\nReplicates per compound:")
print(replicate_counts.describe().round(1))
print(f"\nMin replicates: {replicate_counts.min()}")
print(f"Max replicates: {replicate_counts.max()}")

Metadata columns: 4
Feature columns:  3180

Replicates per compound:
count     175.0
mean      108.4
std       754.4
min         3.0
25%         5.0
50%         8.0
75%        12.0
max      7887.0
dtype: float64

Min replicates: 3
Max replicates: 7887


In [4]:
# identify the outlier compounds

replicate_counts = profiles.groupby('Metadata_JCP2022').size().sort_values(ascending=False)

print('top 10 compounds by replicate count:')
top10 = replicate_counts.head(10).reset_index()
top10.columns = ['Metadata_JCP2022', 'replicates']

# add drug names via compound_lookup
top10 = top10.merge(compound_lookup[['drug_name', 'Metadata_JCP2022']], on='Metadata_JCP2022', how='left')
print(top10.to_string())

print(f"\ncompounds with > 100 replicates: {(replicate_counts > 100).sum()}")
print(f"compounds with < 5 replicates: {(replicate_counts < 5).sum()}")

top 10 compounds by replicate count:
  Metadata_JCP2022  replicates          drug_name
0   JCP2022_046054        7887          Daporinad
1   JCP2022_035095        6173          LY2109761
2   JCP2022_033914         297            AZD7762
3   JCP2022_013856         293         Buparlisib
4   JCP2022_001036         291        Afuresertib
5   JCP2022_105442         290            BI-2536
6   JCP2022_060040         289      5-azacytidine
7   JCP2022_029868         171        Ruxolitinib
8   JCP2022_077046         158          Motesanib
9   JCP2022_071429         155  N-acetyl cysteine

compounds with > 100 replicates: 21
compounds with < 5 replicates: 5


In [6]:
# aggregate replicates into one consensus profile per drug

print("Aggregating replicates to consensus profiles (median)...")

consensus = (
    profiles
    .groupby('Metadata_JCP2022')[feature_cols]
    .median()
    .reset_index()
)

# Add drug names back
consensus = consensus.merge(
    compound_lookup[['drug_name', 'Metadata_JCP2022']],
    on='Metadata_JCP2022',
    how='left'
)

print(f"Consensus profiles shape: {consensus.shape}")
print(f"Unique drugs: {consensus['Metadata_JCP2022'].nunique()}")
print(f"\nSample:")
print(consensus[['drug_name', 'Metadata_JCP2022']].head(5).to_string())

Aggregating replicates to consensus profiles (median)...
Consensus profiles shape: (175, 3182)
Unique drugs: 175

Sample:
           drug_name Metadata_JCP2022
0          I-BET-762   JCP2022_000005
1          Erlotinib   JCP2022_000088
2        Afuresertib   JCP2022_001036
3  alpha-lipoic acid   JCP2022_001180
4        Palbociclib   JCP2022_001418


In [8]:
missing = consensus[feature_cols].isna().sum()
missing_pct = missing / len(consensus) * 100

print(f"features with any missing values: {(missing > 0).sum()}")
print(f"features with >10% missing:       {(missing_pct > 10).sum()}")
print(f"features with >50% missing:       {(missing_pct > 50).sum()}")
print(f"\ntotal missing values: {missing.sum()}")
print(f"overall missingness:  {missing.sum() / (len(consensus) * len(feature_cols)) * 100:.3f}%")

features with any missing values: 0
features with >10% missing:       0
features with >50% missing:       0

total missing values: 0
overall missingness:  0.000%


In [9]:
# qc (check for near-zero variance features)

feature_std = consensus[feature_cols].std()
zero_var = (feature_std == 0).sum()
near_zero_var = (feature_std < 0.01).sum()

print(f"zero variance features: {zero_var}")
print(f"near-zero variance (<0.01): {near_zero_var}")
print(f"\feature std distribution:")
print(feature_std.describe().round(4))

# drop zero variance features
if zero_var > 0:
    cols_to_drop = features_std[feature_std == 0].index.tolist()
    consensus = consensus.drop(columns=cols_to_drop)
    feature_cols = [c for c in feature_cols if c not in cols_to_drop]
    print(f"\ndropped {zero_var} zero-variance features.")
    print(f"remaining feature columns: {len(feature_cols)}")
else:
    print(f"\nno features dropped. all {len(feature_cols)} features retained!")

zero variance features: 0
near-zero variance (<0.01): 0
eature std distribution:
count    3180.0000
mean        0.9842
std         0.1281
min         0.4306
25%         0.8968
50%         0.9859
75%         1.0785
max         1.3791
dtype: float64

no features dropped. all 3180 features retained!


In [10]:
# qc: check for duplicate features

# transpose so features are rows, drugs are columns
feature_matrix = consensus[feature_cols].T

# find duplicate rows (identical feature vectors across all 175 drugs)
duplicated_features = feature_matrix.duplicated()
n_duplicates = duplicated_features.sum()

print(f"duplicate features: {n_duplicates}")

if n_duplicates > 0:
    feature_cols_clean = feature_matrix[~duplicated_features].index.tolist()
    consensus = consensus[['Metadata_JCP2022', 'drug_name'] + feature_cols_clean]
    print(f"dropped {n_duplicates} duplicate features.")
    print(f"remaining: {len(feature_cols_clean)}")
else:
    print(f"no duplicate features. all {len(feature_cols)} retained!")

duplicate features: 2
dropped 2 duplicate features.
remaining: 3178


In [11]:
# generate Morgan fingerprints for all 175 drugs

from rdkit import Chem
from rdkit.Chem import AllChem
import numpy as np

smiles_list = compound_lookup[['drug_name', 'inchikey']].copy()
smiles_list = smiles_list.merge(
    pd.read_csv(PROC_DIR / 'gdsc2_drugs_with_smiles.csv')[['drug_name', 'smiles']],
    on = 'drug_name', how='left'
)

print(f"drugs with SMILES: {smiles_list['smiles'].notna().sum()} / {len(smiles_list)}")

# generate Morgan fingerprints (radius=2, nbits=2048)
def smiles_to_morgan(smiles, radius=2, nbits=2048):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return None
        fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=nbits)
        return np.array(fp)
    except:
        return None

fingerprints = []
failed = []

for _, row in smiles_list.iterrows():
    fp = smiles_to_morgan(row['smiles'])
    if fp is not None:
        fingerprints.append({'drug_name': row['drug_name'], 'morgan_fp': fp})
    else:
        failed.append(row['drug_name'])

print(f"fingerprints generated: {len(fingerprints)}")
print(f"failed: {len(failed)}")
if failed:
    print(f"failed drugs: {failed}")

drugs with SMILES: 175 / 175
fingerprints generated: 175
failed: 0


[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerator
[05:19:13] DEPRECATION WARNING: please use MorganGenerat

In [12]:
# build the Morgan fingerprints to DataFrame

fp_array = np.vstack([f['morgan_fp'] for f in fingerprints])
fp_cols = [f'morgan_{i}' for i in range(2048)]

morgan_df = pd.DataFrame(fp_array, columns=fp_cols)
morgan_df.insert(0, 'drug_name', [f['drug_name'] for f in fingerprints])

print(f"Morgan fingerprint matrix: {morgan_df.shape}")
print(f"\nbit density (mean bits set):")
bit_density = fp_array.mean()
print(f"{bit_density:.3f} ({bit_density*100:.1f}% of bits set per drug on average)")
print(f"\nsample:")
print(morgan_df[['drug_name'] + fp_cols[:5]].head(3).to_string())

Morgan fingerprint matrix: (175, 2049)

bit density (mean bits set):
0.027 (2.7% of bits set per drug on average)

sample:
    drug_name  morgan_0  morgan_1  morgan_2  morgan_3  morgan_4
0   Gefitinib         0         0         0         0         0
1  Vorinostat         0         0         0         0         0
2   Nilotinib         0         0         0         0         0


In [15]:
# redefine feature_cols from actual consensus DataFrame
meta = ['Metadata_JCP2022', 'drug_name']
feature_cols = [c for c in consensus.columns if c not in meta]
print(f"Actual feature columns in consensus: {len(feature_cols)}")

Actual feature columns in consensus: 3178


In [16]:
# save all three features matrices & build the master table

# save Morgan fingerprints
morgan_df.to_csv(PROC_DIR / "morgan_fingerprints.csv", index=False)
print(f"saved: morgan_fingerprints.csv → {morgan_df.shape}")

# save consensus morphological profiles
consensus_out = consensus[['drug_name', 'Metadata_JCP2022'] + feature_cols].copy()
consensus_out.to_csv(PROC_DIR / "consensus_profiles.csv", index=False)
print(f"saved: consensus_profiles.csv → {consensus_out.shape}")

# Build master table: drug_name | morgan_fp (2048) | morphology (3178) | ic50_vector (969)
ic50_reset = ic50_matrix.reset_index()
ic50_reset = ic50_reset.rename(columns={'DRUG_NAME': 'drug_name'})

master = morgan_df.merge(consensus_out[['drug_name'] + feature_cols], on='drug_name', how='inner')
master = master.merge(ic50_reset, on='drug_name', how='inner')

print(f"\nmaster table shape: {master.shape}")
print(f"expected: 175 drugs × (1 + 2048 + 3178 + 969) = 175 × 6196 + drug_name")
print(f"\ndrugs in master table: {master['drug_name'].nunique()}")

saved: morgan_fingerprints.csv → (175, 2049)
saved: consensus_profiles.csv → (175, 3180)

master table shape: (175, 6196)
expected: 175 drugs × (1 + 2048 + 3178 + 969) = 175 × 6196 + drug_name

drugs in master table: 175


In [17]:
# save the master table

master.to_csv(PROC_DIR / 'training_table.csv', index=False)
print(f"saved: training_table.csv -> {master.shape}")
print(f"\processed folder contents:")
for f in sorted(PROC_DIR.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"{f.name:<35} {size_mb:.1f} MB")

saved: training_table.csv -> (175, 6196)
\processed folder contents:
compound_lookup.csv                 0.0 MB
consensus_profiles.csv              6.2 MB
gdsc2_drugs_with_smiles.csv         0.0 MB
gdsc2_ic50_matrix.csv               1.4 MB
gdsc2_jump_overlap.csv              0.0 MB
gdsc2_matched.csv                   23.6 MB
jump_profiles_matched.parquet       357.6 MB
morgan_fingerprints.csv             0.7 MB
training_table.csv                  8.3 MB
